# 037 · RMSprop (Root Mean Square Propagation)

**AdaGrad with the sum replaced by an EWMA.** One line changes:

$$v_t = \beta v_{t-1} + (1-\beta)g_t^2, \qquad w \leftarrow w - \frac{\eta}{\sqrt{v_t}+\varepsilon}g_t$$

| Part | What we reproduce |
|---|---|
| A | one gradient stream, both accumulators: AdaGrad **4.6 → 891**, RMSprop settles near **1** |
| B | AdaGrad's effective rate falls to **7%** of its step-10 value and is still falling |
| C | **RMSprop's accumulator can decrease.** AdaGrad's cannot |
| D | what RMSprop does *not* fix |

Needs `numpy`. This is the whole argument for RMSprop and it needs no model at all.

In [ ]:
import numpy as np

STEPS, LR, EPS, BETA = 1000, 0.1, 1e-8, 0.95
rng = np.random.default_rng(7)
grads = rng.normal(0.0, 1.0, STEPS)     # noise that never vanishes

print(f"{STEPS} gradients, mean {grads.mean():.3f}, std {grads.std():.3f}")
print("The gradient magnitude is STEADY. Nothing about the problem is getting")
print("easier - so an honest step-size rule should also stay steady.")

## Part A — Feed the same stream into both

In [ ]:
acc_sum = acc_ewma = 0.0
rows = {}
for t, g in enumerate(grads, 1):
    acc_sum += g ** 2                                   # AdaGrad: a SUM
    acc_ewma = BETA * acc_ewma + (1 - BETA) * g ** 2    # RMSprop: an EWMA
    if t in (10, 100, 500, 1000):
        rows[t] = (acc_sum, acc_ewma,
                   LR / (np.sqrt(acc_sum) + EPS), LR / (np.sqrt(acc_ewma) + EPS))

print(f"{'step':<8}{'AdaGrad acc':>13}{'RMSprop acc':>13}"
      f"{'AdaGrad lr':>13}{'RMSprop lr':>13}")
for t, (a, r, la, lr_) in rows.items():
    print(f"{t:<8}{a:>13.1f}{r:>13.3f}{la:>13.5f}{lr_:>13.5f}")

In [ ]:
first, last = rows[10], rows[1000]
print(f"AdaGrad: accumulator {first[0]:.1f} -> {last[0]:.1f}")
print(f"         effective rate fell to {last[2]/first[2]:.1%} of its step-10 value")
print(f"RMSprop: accumulator settled near {last[1]:.2f}, effective rate steady")

assert last[0] > 800                    # AdaGrad grew without bound
assert 0.3 < last[1] < 3.0              # RMSprop settled
assert last[2] / first[2] < 0.10        # AdaGrad's rate collapsed

## Part B — A sum has unbounded memory; an EWMA forgets

`v_t` tracks **recent** gradient magnitude. `G_t` tracks **lifetime** gradient
magnitude — which grows with the number of steps taken, not with anything about
the problem.

In [ ]:
print("what each accumulator is really measuring, on constant gradients of 1.0:\n")
acc_sum = acc_ewma = 0.0
for t in range(1, 1001):
    acc_sum += 1.0
    acc_ewma = BETA * acc_ewma + (1 - BETA) * 1.0
    if t in (10, 100, 1000):
        print(f"  step {t:>5}: AdaGrad {acc_sum:>7.1f}   RMSprop {acc_ewma:.4f}")

print("\nThe gradient never changed. AdaGrad's accumulator grew 100x anyway -")
print("it is counting steps. RMSprop converges to the true value, 1.0.")
print(f"\nEffective window: 1/(1-beta) = {1/(1-BETA):.0f} steps.")

## Part C — RMSprop's accumulator can go *down*

This is the property that matters, and it is easy to check.

In [ ]:
acc_sum = acc_ewma = 0.0
history_sum, history_ewma = [], []
for g in grads:
    acc_sum += g ** 2
    acc_ewma = BETA * acc_ewma + (1 - BETA) * g ** 2
    history_sum.append(acc_sum)
    history_ewma.append(acc_ewma)

print(f"RMSprop at step  500 : {history_ewma[499]:.4f}")
print(f"RMSprop at step 1000 : {history_ewma[999]:.4f}")
print(f"  -> it went {'DOWN' if history_ewma[999] < history_ewma[499] else 'up'}")

decreases_ewma = sum(1 for i in range(1, STEPS) if history_ewma[i] < history_ewma[i-1])
decreases_sum = sum(1 for i in range(1, STEPS) if history_sum[i] < history_sum[i-1])
print(f"\nsteps where the accumulator decreased:")
print(f"  RMSprop : {decreases_ewma} of {STEPS-1}")
print(f"  AdaGrad : {decreases_sum} of {STEPS-1}   (mathematically impossible)")

assert decreases_sum == 0
assert decreases_ewma > 300

In [ ]:
# The name is literal: the ROOT of the MEAN of the SQUARED gradients.
recent = grads[-20:]
by_hand = np.sqrt(np.mean(recent ** 2))
print(f"root mean square of the last 20 gradients : {by_hand:.4f}")
print(f"sqrt of RMSprop's accumulator             : {np.sqrt(history_ewma[-1]):.4f}")
print("\nNot identical - the EWMA weights recent points more heavily than a")
print("flat 20-point window does - but it is the same quantity being estimated.")

## Part D — What it does *not* fix

RMSprop scales each parameter's step by recent gradient magnitude. Magnitude has
no sign, so RMSprop has **no memory of direction** at all.

In [ ]:
# Two parameters with identical magnitude but completely different behaviour.
consistent = np.ones(40)                       # always the same way
oscillating = np.array([1.0, -1.0] * 20)       # flip-flopping

for name, stream in (("consistent", consistent), ("oscillating", oscillating)):
    v = 0.0
    for g in stream:
        v = BETA * v + (1 - BETA) * g ** 2
    print(f"  {name:<12} RMSprop accumulator = {v:.4f}")

print("\nIdentical. RMSprop cannot tell these two apart, because it only ever")
print("looks at g^2. Momentum can tell them apart, because it accumulates g.")
print("\nThat is exactly the gap Adam closes: keep BOTH averages.")

## What to take away

- **RMSprop = AdaGrad with the sum replaced by an EWMA.** One line changes.
- **`v_t = βv_{t−1} + (1−β)g_t²`**, then `w ← w − η/(√v_t + ε) g_t`. β ≈ 0.95.
- **A sum has unbounded memory; an EWMA forgets.** `v_t` tracks *recent*
  gradient magnitude, not *lifetime*.
- Measured over 1,000 steps: AdaGrad's accumulator went **4.6 → 891**;
  RMSprop's **settled near 1**.
- AdaGrad's effective rate fell to **7%** of its step-10 value and was still falling.
- **RMSprop's accumulator can decrease** — it was lower at step 1,000 than at
  500. AdaGrad's cannot, and this notebook checks that it never once did.
- **The name is literal:** the **root** of the **mean** of the **squared** gradients.
- **Fixes the per-parameter rate and AdaGrad's decay. Does not fix oscillation or
  saddle points** — no memory of direction.
- Defaults: **β = 0.95, η = 0.001.** Still a good choice for recurrent networks.

## Exercises

1. Sweep β from 0.5 to 0.999. What does a very low β do to the effective rate,
   and why is it a bad idea?
2. Feed a gradient stream whose magnitude genuinely *decreases* over time (as it
   would near a minimum). Which accumulator responds correctly?
3. RMSprop was introduced in a lecture slide, not a paper. Find the original
   description and check whether the β it recommends matches what frameworks use.
4. Re-run lesson 036's sparse-feature problem with RMSprop. Does it match
   AdaGrad's 99% recovery, or does forgetting hurt on sparse data?
5. Part D showed RMSprop cannot distinguish consistent from oscillating
   gradients. Build the smallest problem where that costs it, and confirm
   momentum handles it.